# H4: модель CAMS против станции

**Что такое CAMS.** В `cams_pm25` записаны не натурные измерения датчика, а модельные значения атмосферного реанализа **Copernicus CAMS** (ячейка сетки `43.30° с.ш., 76.90° в.д.`, данные из Open-Meteo Air Quality API; первые непустые значения для Алматы начинаются с `2022-08-04`). Мы сравниваем эту модель с эталонной станцией **OpenAQ ID 8876** (Алматы, провайдер AirNow).

**Формулировки гипотез (до расчётов):**
- **H0:** в среднем CAMS совпадает со станцией (`bias = 0`, где `bias = среднее(CAMS − станция)`).
- **H1:** CAMS систематически отличается от станции (`bias ≠ 0`).

**Как принимаем решение:** оцениваем **95 % bootstrap доверительный интервал (CI)** суточного `bias` (`α = 0.05` ↔ 95 % CI). Если CI не содержит `0`, отвергаем H0. Дополнительно оцениваем размер эффекта (`bias` в µg/m³ и %), `MAE`, корреляции Пирсона `r` и Спирмена `ρ` и строим график сравнения. Гипотеза H4 проверяется по доверительному интервалу и является отдельной от подтверждающих p-тестов (`m = 5`) из `notebooks/03_eda.ipynb`.

**Почему главный уровень — сутки, а часы — дополнительно:** на часах `MAE` и корреляция сильно зависят от того, насколько точно совпадают суточные циклы станции и модели, которые сдвинуты друг относительно друга на несколько часов. Суточное среднее сглаживает внутрисуточный сдвиг на 1–3 ч, а `bias` (разница средних) к такому сдвигу почти нечувствителен, поэтому `bias` оцениваем на обоих уровнях, но основные выводы по H4 и согласию рядов делаем по суткам.

**Оговорки по времени (ADR-017):**
- Время CAMS — гарантированный **UTC**. Метки станции анализируются **при допущении, что они тоже в UTC**: версия сдвига `−6` ч не исключена, а сдвиг на `1–3` ч нашими проверками не различим.
- Сравниваем ряды строго на **лаге 0** по `time_utc` и ничего не сдвигаем во времени: лаговая проверка уже проведена в `notebooks/03_eda.ipynb` (часть 2, шаг 16, график 12). За весь период с учётом интерполированных часов (`n = 25 780`) на лаге `0` получено `ρ = 0.534`, на лучшем лаге `−2` ч — `ρ = 0.556`, на `+6` ч — `0.377`, а при 24-часовом сглаживании на лаге `0` — `0.770` (4.2, `03_eda`, шаг 16). Подбирать сдвиг под максимум корреляции на тех же данных означало бы подгонять результат задним числом, поэтому здесь лаги повторно не пересчитываем.

In [1]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append("..")
from src import cams, clean, features, hypotheses

HOURLY_PATH = "../data/processed/hourly.csv"
DAILY_PATH = "../data/processed/daily.csv"
FIGURES_DIR = "../figures"

HEATING_LABELS = {True: "Отопительный сезон", False: "Вне сезона"}

# Единая тема графиков: палитра, удобная для дальтоников
sns.set_theme(style="whitegrid", context="talk", palette="colorblind")

## Шаг 1. Данные и константы

Загружаем очищенные таблицы `hourly.csv` и `daily.csv` из `data/processed/`, проверяем их структуру (`check_hourly`, `check_daily`) и задаём общие константы: число bootstrap-выборок, фиксированный `SEED` и шаг прореживания суток для проверки устойчивости к автокорреляции.

In [2]:
hourly = clean.read_clean_csv(HOURLY_PATH)
features.check_hourly(hourly)

daily = features.read_daily_csv(DAILY_PATH)
features.check_daily(daily)

# Параметры bootstrap (как в части 3 ноутбука 03_eda)
N_BOOT = 5000
SEED = 42
# Тот же шаг прореживания суток, что выбран в задаче 5.1 (03_eda, шаг 20)
K_THIN = 14

print(f"Строк в hourly: {len(hourly)} ({hourly['time_utc'].min()} … {hourly['time_utc'].max()})")
print(f"Строк в daily:  {len(daily)} ({daily['date'].min().date()} … {daily['date'].max().date()})")

Строк в hourly: 49075 (2020-04-09 16:00:00+00:00 … 2025-11-14 10:00:00+00:00)
Строк в daily:  2044 (2020-04-10 … 2025-11-13)


## Шаг 2. Часы, где есть оба значения

Для проверки H4 берём только реальные измеренные часы станции: строки, где заполнены и `pm25`, и `cams_pm25`, а флаг `pm25_interpolated == False`. Заполненные на этапе очистки короткие пропуски (до 3 часов) не являются натурными измерениями и в сравнение модели со станцией не входят.

Повторный `merge` на часовом уровне не нужен: оба ряда уже объединены в `hourly.csv` по ключу `time_utc` на этапе 3.1 (`validate="one_to_one"`). Дополнительно считаем коэффициент Спирмена на всех строках с обеими величинами (включая интерполированные), чтобы сверить данные с результатом из шага 16 `03_eda.ipynb`.

In [3]:
has_cams = hourly["cams_pm25"].notna()
both_with_interp = hourly["pm25"].notna() & has_cams
excluded_interp = both_with_interp & hourly["pm25_interpolated"].astype(bool)

print(f"Всего часов в hourly: {len(hourly)}")
print(f"Часов с непустым cams_pm25: {int(has_cams.sum())}")
print(f"Часов с обоими значениями (вкл. интерполированные): {int(both_with_interp.sum())}")
print(f"Исключено интерполированных часов станции: {int(excluded_interp.sum())}")

pairs = cams.paired_hours(hourly)
assert pairs["time_utc"].min() >= pd.Timestamp("2022-08-04", tz="UTC")

in_season = int(pairs["heating_season"].sum())
out_season = int((~pairs["heating_season"]).sum())
print(f"Итог парных измеренных часов (pairs): n = {len(pairs)}")
print(f"Период пар (UTC): {pairs['time_utc'].min()} … {pairs['time_utc'].max()}")
print(f"В отопительный сезон: {in_season} | Вне сезона: {out_season}")

# Сверка с шагом 16 (4.2, 03_eda): ρ Спирмена с интерполированными часами на лаге 0
check_with_interp = cams.agreement_metrics(hourly.loc[both_with_interp])
print(
    f"Сверка с 4.2 (с интерполированными): n = {check_with_interp['n']}, "
    f"spearman_rho = {check_with_interp['spearman_rho']:.6f}"
)

Всего часов в hourly: 49075
Часов с непустым cams_pm25: 28763
Часов с обоими значениями (вкл. интерполированные): 25780
Исключено интерполированных часов станции: 222
Итог парных измеренных часов (pairs): n = 25558
Период пар (UTC): 2022-09-06 18:00:00+00:00 … 2025-11-14 10:00:00+00:00
В отопительный сезон: 12191 | Вне сезона: 13367
Сверка с 4.2 (с интерполированными): n = 25780, spearman_rho = 0.533752


**Что видим.** (вывод шага 2) Из `49 075` часов таблицы `hourly` прогноз CAMS заполнен в `28 763` строках, а оба значения (включая интерполированные часы станции) есть в `25 780` часах. После исключения `222` интерполированных часов для проверки H4 остаётся `25 558` парных измеренных часов за период `2022-09-06 18:00 UTC … 2025-11-14 10:00 UTC`: `12 191` час в отопительный сезон и `13 367` часов вне сезона. Контрольный расчёт по всем `25 780` часам с интерполяцией даёт `spearman_rho = 0.533752`, что совпадает со значением из шага 16 `03_eda.ipynb` (`0.534`, `n = 25 780`).

## Шаг 3. CAMS и станция по часам

Оцениваем согласие модели CAMS со станцией на парных измеренных часах за весь период, в отопительный сезон и вне его.

**Метрики:**
- `bias = среднее(CAMS − станция)` в µg/m³ (и `bias_pct` — доля от среднего станции в %). Отрицательный знак означает, что CAMS в среднем занижает концентрацию PM2.5, положительный — завышает.
- `MAE = среднее(|CAMS − станция|)` в µg/m³ — средняя абсолютная ошибка без взаимного погашения знаков.
- Корреляции **Пирсона `r`** (линейная связь, как в постановке задачи) и **Спирмена `ρ`** (ранговая связь, устойчивая к редким пиковым выбросам PM2.5).

Доверительные интервалы на часовом уровне не строим: соседние часы сильно похожи друг на друга, поэтому bootstrap по ~25 тыс. часов дал бы обманчиво узкий интервал. Кроме того, почасовые `MAE` и `ρ` чувствительны к совмещению суточных циклов станции и модели (ADR-017: метки станции анализируем при допущении UTC, возможен сдвиг фазы), поэтому часовой уровень считаем дополнительным, а основные выводы делаем по суткам в шаге 4.

In [4]:
GROUPS = {
    "весь период": pd.Series(True, index=pairs.index),
    "отопительный сезон": pairs["heating_season"],
    "вне сезона": ~pairs["heating_season"],
}

hourly_rows = []
for group_name, mask in GROUPS.items():
    metrics = cams.agreement_metrics(pairs.loc[mask])
    hourly_rows.append({"grain": "часы", "group": group_name, **metrics})

hourly_table = pd.DataFrame(hourly_rows)
print(hourly_table.round(3).to_string(index=False))

grain              group     n  station_mean  cams_mean    bias  bias_pct    mae  pearson_r  spearman_rho
 часы        весь период 25558        26.435     21.971  -4.464   -16.886 15.878      0.504         0.533
 часы отопительный сезон 12191        41.651     28.635 -13.015   -31.249 25.845      0.402         0.442
 часы         вне сезона 13367        12.557     15.892   3.335    26.562  6.788      0.276         0.341


**Что видим.** (вывод шага 3)
- **Весь период** (`n = 25 558`): среднее станции `26.435` µg/m³, CAMS `21.971` µg/m³; `bias = -4.464` µg/m³ (`-16.886 %`), `MAE = 15.878` µg/m³, `r = 0.504`, `ρ = 0.533`. Часовая ранговая корреляция `ρ = 0.533` практически совпадает со значением из 4.2 (`0.534` на `25 780` часах с интерполированными, сверено в шаге 2): исключение интерполированных часов почти не меняет оценку связи.
- **Отопительный сезон** (`n = 12 191`): среднее станции `41.651` µg/m³, CAMS `28.635` µg/m³; CAMS занижает на `bias = -13.015` µg/m³ (`-31.249 %`), `MAE = 25.845` µg/m³, `r = 0.402`, `ρ = 0.442`.
- **Вне сезона** (`n = 13 367`): среднее станции `12.557` µg/m³, CAMS `15.892` µg/m³; знак смещения меняется — CAMS завышает на `bias = 3.335` µg/m³ (`26.562 %`), `MAE = 6.788` µg/m³, `r = 0.276`, `ρ = 0.341`.

При противоположных знаках по сезонам (`-13.015` µg/m³ в отопительный сезон и `+3.335` µg/m³ вне его) общий `bias` (`-4.464` µg/m³) частично сглаживается, тогда как `MAE` за весь период достигает `15.878` µg/m³.